# Building Blocks of Neural Networks

## Backpropagation using Gradient Tape

**Backpropagation** is the process to efficiently calculate the gradients of the loss function with respect to the weights and gradient descent is used to update the weights. 
Backpropagation keeps repeating the chain rule to calculate the gradients. 

Chain Rule: A variable `z` depends on the variable `y`, which itself depends on variable `x` (i.e., variable $y$ and $z$ are dependent variables), then $z$ depends on $x$ as well, via the intermediate variable $y$.

$$
\frac{∂z}{∂x} = \frac{∂z}{∂y} \space * \space \frac{∂y}{∂x}
$$


Neural Networks training process:
1. Input: Input comprises of random set of weights and a dataset of images to train the NN.
1. Forward Pass: The NN goes through the different layers, activation functions and biases. 
1. Prediction: The NN predicts the output based on the current weights, dataset and the model. 
1. Calculate Loss: Calculate the loss based on the predictions and true values. 
1. Compute Gradient: Calculate the gradients of the loss with respect to the current weights and layer type.
1. Backward Propagation: 
1. Update Weights: Update the weights using gradient descent. Repeast from Step 1 with the new wights.

- Backpropagation algorithm is used heavily in neural networks to update the model's parameters.
- The algorithm works by continuously moving backwards in the network, finding the partial derivatives of the loss function w.r.t. the model's parameters, and then performing the parameter updates.
- A key task in backpropagation is to first find out the gradient values for each trainable parameter of the model.
- We know that Backpropagation algorithm uses Chain rule to find the partial derivatives.
- But to derive and define the gradient calculation for each parameter on our own can be tedious, there being so many parameters.
- However, when using modern Deep Learning frameworks such as Tensorflow, PyTorch or MXNet, we generally don't have to worry about calculating these gradients manually. It’s done automatically for us.

### Gradient Tape

From Tensorflow:

> TensorFlow provides the tf.GradientTape API for automatic differentiation; that is, computing the gradient of a computation with respect to some inputs, usually tf.Variables. TensorFlow "records" relevant operations executed inside the context of a tf.GradientTape onto a "tape".
TensorFlow then uses that tape to compute the gradients of a "recorded" computation using reverse mode differentiation."

Within the with `tf.GradientTape() as tape` context manager we'll perform some operations.

> "To differentiate automatically, TensorFlow needs to remember what operations happen in what order during the forward pass. Then, during the backward pass, TensorFlow traverses this list of operations in reverse order to compute gradients.

In [1]:
import tensorflow as tf

In [2]:
x = tf.Variable(2.0)

# Now that GradientTape has recorded the operation
# we can calculate the gradient of the operation i.e. dy/dx
with tf.GradientTape() as tape:
    y = x**2

# Note: We are now outside the GradientTape context
# Gradient calculations and updates need to be performed
# outside the GradientTape context, or these operations will be
# recorded on the tape as well, and increased memory usage.
dy_dx = tape.gradient(y, x)

# Gradient value
print(dy_dx.numpy())

4.0


#### Check the value manually

$$
y = x^2 
$$
$$
\frac{dy}{dx} = \frac{d{x^2}}{dx}
$$
$$
Derivative \space of \space x^n = n * x^{n-1}
$$
$$
\frac{dy}{dx} = 2 * \frac{d{x}}{dx}
$$
$$
\frac{dy}{dx} = 2 * x
$$
$$
\frac{dy}{dx} = 2 * 2
$$
$$
\frac{dy}{dx} = 4
$$

### GradientTape Params

The tf.GradientTape class takes in 2 parameters, which are as follows:

- watch_accessed_variables: (Boolean, Default: True) Controls whether the tape will automatically watch any (trainable) variables that are accessed while the tape is active. This means gradients can be requested from any result computed in the tape derived from reading a trainable Variable. If False, users must explicitly watch any Variables they want to request gradients from.
- persistent: (Boolean, Default: False) Controls whether or not to create a persistent gradient tape. This should be used when you need to compute more than one set of gradients.

## Setup:

#### Equations:

$$
A_1 = x_1 * w_1 + x_2 * w_2 + b 
$$

$$
A_2 = sigmoid(A_1)
$$

$$
A_3 = J(A_2, Y)
$$

### Manual Update

In [3]:
# Define the constants and variables
x1 = tf.constant(1.3, name="x1")
x2 = tf.constant(2.1, name="x2")
lr = tf.constant(0.1, name="learning_rate")
Y = tf.constant(1.0, name="ground_truth")

w1 = tf.Variable(0.7, "w1")
w2 = tf.Variable(-0.3, "w2")
b = tf.Variable(1.0, "bias")

In [4]:
# Implementation for equation A1 and its derivative


def wx_plus_b(
    x1: tf.constant, w1: tf.Variable, x2: tf.constant, w2: tf.Variable, b: tf.Variable
) -> tf.Variable:
    return x1 * w1 + x2 * w2 + b


def derivative_wx_plus_b(x1, x2):
    return x1, x2, tf.constant(1.0)


# Implementation of equation A2 and its derivative


def sigmoid(x) -> tf.Variable:
    return 1 / (1 + tf.math.exp(-x))


def derivative_sigmoid(x):
    return sigmoid(x) * (1.0 - sigmoid(x))


# Implementation of equation A3 and its derivative


def bce_loss(y_hat, y) -> tf.Variable:
    return -(y * tf.math.log(y_hat)) - ((1 - y) * tf.math.log(1.0 - y_hat))


def derivative_bce_loss(y_hat, y):
    return -(y / y_hat) + ((1.0 - y) / (1.0 - y_hat))

In [5]:
# In the forward function, the input data is passed through the equations one by one in a forward manner.
def forward(x1, x2, w1, w2, b, Y):
    A1 = wx_plus_b(x1, w1, x2, w2, b)
    A2 = sigmoid(A1)
    A3 = bce_loss(y_hat=A2, y=Y)

    return {
        "A1": A1,
        "A2": A2,
        "A3": A3,
    }

In [6]:
# The backward function is responsible for calculating the derivative of `A3` w.r.t. the variables `w_1, w_2, b`.
# In this function, we first calculate the gradient values for each partial derivative involved in the chain rule equation and then we use them to solve the chain rule.


def backward(x1, x2, Y, A1, A2):
    # Compute the gradients of A3 w.r.t A2, i.e., dA3/dA2
    d_bce_loss = derivative_bce_loss(y_hat=A2, y=Y)

    # Compute the gradients of A2 w.r.t A1, i.e., dA2/dA1
    d_sigmoid = derivative_sigmoid(A1)

    # Compute the gradients of weighted sums(A1) w.r.t weights and bias
    # i.e., dA1/aw1, dA1/dw2, dA1,db
    d_w1, d_w2, d_b = derivative_wx_plus_b(x1, x2)

    # Use chain rule to find overall gradient of Loss w.r.t weights and bias
    w1_grad = d_bce_loss * d_sigmoid * d_w1
    w2_grad = d_bce_loss * d_sigmoid * d_w2
    b_grad = d_bce_loss * d_sigmoid * d_b

    return {
        "dA3_dA2": d_bce_loss,
        "dA2_dA1": d_sigmoid,
        "dA1_dw1": d_w1,
        "dA1_dw2": d_w2,
        "dA1_db": d_b,
        "dA3_dw1": w1_grad,
        "dA3_dw2": w2_grad,
        "dA3_db": b_grad,
    }

In [7]:
# Execute the forward function to get the initial Loss.
forward_outputs = forward(x1, x2, w1, w2, b, Y)

print(f"Forward Pass:\n")

print(f"A1: {forward_outputs['A1']}")
print(f"A2: {forward_outputs['A2']}")
print(f"A3: {forward_outputs['A3']} <--- Initial Loss")

Forward Pass:

A1: 1.2799999713897705
A2: 0.7824497818946838
A3: 0.24532553553581238 <--- Initial Loss


In [8]:
# Execute the backward function to get derivative of Loss w.r.t. to the variables.
A1 = forward_outputs["A1"]
A2 = forward_outputs["A2"]

backward_outputs = backward(x1, x2, Y, A1, A2)

print(f"Backward Pass: Step 1\n")
print(f"Individual Derivatives:\n")

print(f"dA3/dA2 = {backward_outputs['dA3_dA2']}")
print(f"dA2/dA1 = {backward_outputs['dA2_dA1']}")
print(f"dA1/dw1 = {backward_outputs['dA1_dw1']}")
print(f"dA1/dw2 = {backward_outputs['dA1_dw2']}")
print(f"dA1/db  = {backward_outputs['dA1_db']}")

print("\n-----------------\n")

print(f"Gradient of A3 w.r.t. variables:\n")

print(f"dA3/dw1 = {backward_outputs['dA3_dw1']}")
print(f"dA3/dw2 = {backward_outputs['dA3_dw2']}")
print(f"dA3/db  = {backward_outputs['dA3_db']}")

Backward Pass: Step 1

Individual Derivatives:

dA3/dA2 = -1.2780373096466064
dA2/dA1 = 0.17022211849689484
dA1/dw1 = 1.2999999523162842
dA1/dw2 = 2.0999999046325684
dA1/db  = 1.0

-----------------

Gradient of A3 w.r.t. variables:

dA3/dw1 = -0.28281527757644653
dA3/dw2 = -0.456855446100235
dA3/db  = -0.21755021810531616


In [9]:
# Perform the weight updates.
# The weight_update function applies the weight-update rule to change the weight values during backpropagation.
def weight_update(
    w1: tf.Variable,
    w2: tf.Variable,
    b: tf.Variable,
    dw1: tf.Variable,
    dw2: tf.Variable,
    db: tf.Variable,
    lr: tf.constant,
):
    # Since w1, w2 and b are objects of tf.Variable class, they are updated in place.

    # w1 = w1 - lr*dw1
    w1.assign_sub(lr * dw1)

    # w2 -= lr * dw2
    w2.assign_sub(lr * dw2)

    # b -= lr * db
    b.assign_sub(lr * db)

    return w1, w2, b

In [10]:
w1_grad = backward_outputs["dA3_dw1"]
w2_grad = backward_outputs["dA3_dw2"]
b_grad = backward_outputs["dA3_db"]

# keeping a copy of old w and b for comparison
# as w and b will be updated inplace

w1_old = tf.identity(w1, name="old_w1")
w2_old = tf.identity(w2, name="old_w2")
b_old = tf.identity(b, name="old_b")

# Perform Weight Update
w1_updated, w2_updated, b_updated = weight_update(
    w1, w2, b, w1_grad, w2_grad, b_grad, lr
)

print(f"Backward Pass: Step 2\n")
print(f"Parameter Updates\n")

print(f"w1 --> Old: {w1_old.numpy():<20} New: {w1_updated.numpy()}")
print(f"w2 --> Old: {w2_old.numpy():<20} New: {w2_updated.numpy()}")
print(f"b  --> Old: {b_old.numpy():<19}  New: {b_updated.numpy()}")

Backward Pass: Step 2

Parameter Updates

w1 --> Old: 0.699999988079071    New: 0.7282814979553223
w2 --> Old: -0.30000001192092896 New: -0.25431448221206665
b  --> Old: 1.0                  New: 1.0217549800872803


In [11]:
# Compare old loss and new loss

new_forward_outputs = forward(x1, x2, w1_updated, w2_updated, b_updated, Y)

old_A3 = forward_outputs["A3"]
new_A3 = new_forward_outputs["A3"]

print(f"Checking New Loss:\n")

print(f"LOSS --> Old: {old_A3.numpy():<20} New: {new_A3.numpy()}")

Checking New Loss:

LOSS --> Old: 0.24532553553581238  New: 0.21369412541389465


### Using GradientTape

In [12]:
# Define the constants and variables
x1 = tf.constant(1.3, name="x1")
x2 = tf.constant(2.1, name="x2")
lr = tf.constant(0.1, name="learning_rate")
Y = tf.constant(1.0, name="ground_truth")

w1 = tf.Variable(0.7, "w1")
w2 = tf.Variable(-0.3, "w2")
b = tf.Variable(1.0, "bias")

In [13]:
# Set `persistent=True` because the `.gradient(...)` will be called more than once.
with tf.GradientTape(persistent=True) as tape:
    # record operations
    A1 = wx_plus_b(x1, w1, x2, w2, b)
    A2 = sigmoid(A1)
    A3 = bce_loss(A2, Y)

In [14]:
print(f"Forward Pass:\n")

print(f"A1: {A1}")
print(f"A2: {A2}")
print(f"A3: {A3} <--- Initial Loss")

Forward Pass:

A1: 1.2799999713897705
A2: 0.7824497818946838
A3: 0.24532553553581238 <--- Initial Loss


In [15]:
print(f"Backward Pass: Step 1\n")
print(f"Individual Derivatives:\n")


dA3_dA2 = tape.gradient(A3, A2)
dA2_dA1 = tape.gradient(A2, A1)
dA1_dw1 = tape.gradient(A1, w1)
dA1_dw2 = tape.gradient(A1, w2)
dA1_db = tape.gradient(A1, b)

print(f"dA3/dA2 = {dA3_dA2}")
print(f"dA2/dA1 = {dA2_dA1}")
print(f"dA1/dw1 = {dA1_dw1}")
print(f"dA1/dw2 = {dA1_dw2}")
print(f"dA1/db =  {dA1_db}")

print("\n-----------------\n")

# implementing Chain rule
dA3_dw1 = dA3_dA2 * dA2_dA1 * dA1_dw1
dA3_dw2 = dA3_dA2 * dA2_dA1 * dA1_dw2
dA3_db = dA3_dA2 * dA2_dA1 * dA1_db

print(f"Gradient of A3 wrt. variables:\n")

print(f"dA3/dw1 = {dA3_dw1}")
print(f"dA3/dw2 = {dA3_dw2}")
print(f"dA3/db  = {dA3_db}")

Backward Pass: Step 1

Individual Derivatives:

dA3/dA2 = -1.2780373096466064
dA2/dA1 = 0.17022213339805603
dA1/dw1 = 1.2999999523162842
dA1/dw2 = 2.0999999046325684
dA1/db =  1.0

-----------------

Gradient of A3 wrt. variables:

dA3/dw1 = -0.2828153073787689
dA3/dw2 = -0.4568554759025574
dA3/db  = -0.21755023300647736


In [16]:
# keeping a copy of old w and b for comparison
# as w and b will be updated inplace

w1_old = tf.identity(w1, name="old_w1")
w2_old = tf.identity(w2, name="old_w2")
b_old = tf.identity(b, name="old_b")

# Perform Weight Update

w1_updated, w2_updated, b_updated = weight_update(
    w1, w2, b, dA3_dw1, dA3_dw2, dA3_db, lr
)

print(f"Backward Pass: Step 2\n")
print(f"Parameter Updates\n")

print(f"w1 --> Old: {w1_old.numpy():<20} New: {w1_updated.numpy()}")
print(f"w2 --> Old: {w2_old.numpy():<20} New: {w2_updated.numpy()}")
print(f"b  --> Old: {b_old.numpy():<19}  New: {b_updated.numpy()}")

Backward Pass: Step 2

Parameter Updates

w1 --> Old: 0.699999988079071    New: 0.7282814979553223
w2 --> Old: -0.30000001192092896 New: -0.25431445240974426
b  --> Old: 1.0                  New: 1.0217549800872803


In [17]:
# New loss computation

new_forward_outputs = forward(x1, x2, w1_updated, w2_updated, b_updated, Y)

old_A3 = forward_outputs["A3"]
new_A3 = new_forward_outputs["A3"]

# We can also pass w1, w2, b due to the objects being replaced in the memory
# _, _, new_loss = forward(x1, x2, w1, w2, b, Y)

print(f"Checking New Loss:\n")

print(f"LOSS --> Old: {old_A3.numpy():<20} New: {new_A3.numpy()}")

Checking New Loss:

LOSS --> Old: 0.24532553553581238  New: 0.21369412541389465


### Direct Implementation using GradientTape

In [18]:
# Define the constants and variables
x1 = tf.constant(1.3, name="x1")
x2 = tf.constant(2.1, name="x2")
lr = tf.constant(0.1, name="learning_rate")
Y = tf.constant(1.0, name="ground_truth")

w1 = tf.Variable(0.7, "w1")
w2 = tf.Variable(-0.3, "w2")
b = tf.Variable(1.0, "bias")

In [19]:
def compute(x1, x2, w1, w2, b, Y):
    with tf.GradientTape() as tape:
        outputs = forward(x1, x2, w1, w2, b, Y)

    grads = tape.gradient(outputs["A3"], [w1, w2, b])
    return outputs, grads

In [20]:
forward_outputs, gradients = compute(x1, x2, w1, w2, b, Y)

print(f"Backward Pass: Step 1\n")
print(f"Direct Gradient of A3 wrt. variables using GradientTape:\n")

print(f"dA3/dw1 = {gradients[0]}")
print(f"dA3/dw2 = {gradients[1]}")
print(f"dA3/db  = {gradients[2]}")

Backward Pass: Step 1

Direct Gradient of A3 wrt. variables using GradientTape:

dA3/dw1 = -0.2828153073787689
dA3/dw2 = -0.45685550570487976
dA3/db  = -0.21755024790763855


In [21]:
# keeping a copy of old w and b for comparison
# as w and b will be updated inplace

w1_old = tf.identity(w1, name="old_w1")
w2_old = tf.identity(w2, name="old_w2")
b_old = tf.identity(b, name="old_b")

# Perform Weight Update

w1_updated, w2_updated, b_updated = weight_update(
    w1, w2, b, gradients[0], gradients[1], gradients[2], lr
)

print(f"Backward Pass: Step 2\n")
print(f"Parameter Updates\n")

print(f"w1 --> Old: {w1_old.numpy():<20} New: {w1_updated.numpy()}")
print(f"w2 --> Old: {w2_old.numpy():<20} New: {w2_updated.numpy()}")
print(f"b  --> Old: {b_old.numpy():<19}  New: {b_updated.numpy()}")

Backward Pass: Step 2

Parameter Updates

w1 --> Old: 0.699999988079071    New: 0.7282814979553223
w2 --> Old: -0.30000001192092896 New: -0.25431445240974426
b  --> Old: 1.0                  New: 1.0217549800872803


In [22]:
# New loss computation

new_forward_outputs = forward(x1, x2, w1_updated, w2_updated, b_updated, Y)

old_A3 = forward_outputs["A3"]
new_A3 = new_forward_outputs["A3"]

# We can also pass w1, w2, b due to the objects being replaced in the memory
# _, _, new_loss = forward(x1, x2, w1, w2, b, Y)

print(f"Checking New Loss:\n")

print(f"LOSS --> Old: {old_A3.numpy():<20} New: {new_A3.numpy()}")

Checking New Loss:

LOSS --> Old: 0.24532553553581238  New: 0.21369412541389465
